In [1]:
displayHTML("""
<div style="background: linear-gradient(135deg, #1B3A4B 0%, #2D5F7B 100%); padding: 40px 30px; border-radius: 16px; color: #FFFFFF; font-family: 'Segoe UI', Arial, sans-serif; margin-bottom: 20px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <h1 style="font-size: 2.4em; margin: 0 0 10px 0; font-weight: 700; color: #FFFFFF;">Lab 07 &mdash; Build a Delta Live Tables Pipeline</h1>
  <p style="font-size: 1.15em; margin: 0 0 24px 0; opacity: 0.92; color: #FFFFFF;">Azure Databricks Training &bull; Day 2</p>
  <hr style="border: none; border-top: 1px solid rgba(255,255,255,0.3); margin: 0 0 18px 0;">
  <h3 style="margin: 0 0 12px 0; font-weight: 600; color: #FFFFFF;">Learning Objectives</h3>
  <ul style="font-size: 1.05em; line-height: 1.9; margin: 0; padding-left: 20px; color: #FFFFFF;">
    <li>Understand the difference between imperative ETL and declarative DLT pipelines</li>
    <li>Define Bronze, Silver, and Gold tables using the Python DLT API</li>
    <li>Use <strong>data quality expectations</strong> to validate data inline</li>
    <li>Create, configure, and run a DLT pipeline from the Databricks UI</li>
    <li>Monitor pipeline execution and inspect the auto-generated lineage DAG</li>
  </ul>
</div>
""")

Lab 07 — Build a Delta Live Tables Pipeline 
 Azure Databricks Training • Day 2 
 
 Learning Objectives 
 
 Understand the difference between imperative ETL and declarative DLT pipelines 
 Define Bronze, Silver, and Gold tables using the Python DLT API 
 Use data quality expectations to validate data inline 
 Create, configure, and run a DLT pipeline from the Databricks UI 
 Monitor pipeline execution and inspect the auto-generated lineage DAG

---

## Important: How This Lab Works

In [1]:
displayHTML("""
<div style="background: #FFF3E0; border-left: 5px solid #FF3621; padding: 15px 20px; border-radius: 6px; margin: 15px 0; font-family: 'Segoe UI', Arial, sans-serif; color: #1B1F24;">
  <strong style="color: #FF3621;">This notebook is NOT run cell-by-cell like the other labs.</strong><br><br>
  This notebook <em>is</em> the DLT pipeline definition. The entire notebook is executed by the <strong>DLT runtime</strong> when you create and start a pipeline in the Databricks UI.<br><br>
  <strong>Your workflow:</strong>
  <ol style="margin-bottom: 0;">
    <li>Read through the notebook and <strong>fill in the blanks</strong> in the code cells</li>
    <li>Follow the instructions in <strong>Section 6</strong> to create a DLT pipeline in the UI</li>
    <li>Run the pipeline &mdash; if it fails, fix the blanks and re-run</li>
    <li>Inspect the results in the pipeline DAG and event log</li>
  </ol>
</div>
""")

This notebook is NOT run cell-by-cell like the other labs. 
 This notebook is the DLT pipeline definition. The entire notebook is executed by the DLT runtime when you create and start a pipeline in the Databricks UI. 
 Your workflow: 
 
 Read through the notebook and fill in the blanks in the code cells 
 Follow the instructions in Section 6 to create a DLT pipeline in the UI 
 Run the pipeline — if it fails, fix the blanks and re-run 
 Inspect the results in the pipeline DAG and event log

---

## Section 1: What is Delta Live Tables?

In Labs 04–06, you built a Medallion pipeline **imperatively**: you wrote every step yourself — read data, add audit columns, filter nulls, join, aggregate, write to Delta. You managed checkpoints, schema evolution, and orchestration manually.

**Delta Live Tables (DLT)** flips this around. Instead of writing the *how*, you declare the *what*:

In [1]:
displayHTML("""
<div style="display: flex; gap: 20px; flex-wrap: wrap; margin: 20px 0; font-family: 'Cascadia Code', Consolas, monospace; font-size: 13px;">
  <div style="flex: 1; min-width: 300px; background: #1B3A4B; color: #F9F7F4; border-radius: 10px; padding: 18px; line-height: 1.7;">
    <div style="color: #FF6F47; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">IMPERATIVE (Labs 04–06)</div>
    <span style="color: #9E9E9E;">raw_df</span> = spark.readStream<br>
    &nbsp;&nbsp;.format(<span style="color: #81C784;">"cloudFiles"</span>)<br>
    &nbsp;&nbsp;.option(<span style="color: #81C784;">"cloudFiles.format"</span>, ...)<br>
    &nbsp;&nbsp;.load(path)<br>
    <span style="color: #9E9E9E;">bronze_df</span> = raw_df<br>
    &nbsp;&nbsp;.withColumn(<span style="color: #81C784;">"_ingest"</span>, ...)<br>
    bronze_df.writeStream<br>
    &nbsp;&nbsp;.format(<span style="color: #81C784;">"delta"</span>)<br>
    &nbsp;&nbsp;.option(<span style="color: #81C784;">"checkpointLocation"</span>, ...)<br>
    &nbsp;&nbsp;.start(bronze_path)<br><br>
    <span style="color: #FF6F47;"># You manage EVERYTHING</span>
  </div>
  <div style="flex: 1; min-width: 300px; background: #00A972; color: #FFFFFF; border-radius: 10px; padding: 18px; line-height: 1.7;">
    <div style="color: #FFFDE7; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">DECLARATIVE (DLT)</div>
    <span style="color: #FFFDE7;">@dlt.table()</span><br>
    <span style="color: #E0E0E0;">def</span> <span style="color: #FFFDE7;">bronze</span>():<br>
    &nbsp;&nbsp;<span style="color: #E0E0E0;">return</span> spark.readStream<br>
    &nbsp;&nbsp;&nbsp;&nbsp;.format(<span style="color: #C8E6C9;">"cloudFiles"</span>)<br>
    &nbsp;&nbsp;&nbsp;&nbsp;.load(path)<br><br>
    <span style="color: #FFFDE7;"># DLT handles:</span><br>
    <span style="color: #C8E6C9;">#&nbsp;&nbsp;- Checkpoints</span><br>
    <span style="color: #C8E6C9;">#&nbsp;&nbsp;- Schema evolution</span><br>
    <span style="color: #C8E6C9;">#&nbsp;&nbsp;- OPTIMIZE &amp; VACUUM</span><br>
    <span style="color: #C8E6C9;">#&nbsp;&nbsp;- Error recovery</span><br>
    <span style="color: #C8E6C9;">#&nbsp;&nbsp;- Lineage &amp; monitoring</span>
  </div>
</div>
""")

IMPERATIVE (Labs 04–06) 
 raw_df = spark.readStream 
   .format( "cloudFiles" ) 
   .option( "cloudFiles.format" , ...) 
   .load(path) 
 bronze_df = raw_df 
   .withColumn( "_ingest" , ...) 
 bronze_df.writeStream 
   .format( "delta" ) 
   .option( "checkpointLocation" , ...) 
   .start(bronze_path) 
 # You manage EVERYTHING 
 
 
 DECLARATIVE (DLT) 
 @dlt.table() 
 def bronze (): 
    return spark.readStream 
     .format( "cloudFiles" ) 
     .load(path) 
 # DLT handles: 
 #  - Checkpoints 
 #  - Schema evolution 
 #  - OPTIMIZE & VACUUM 
 #  - Error recovery 
 #  - Lineage & monitoring

DLT automatically:
- Builds the **DAG** (dependency graph) from your table references
- Manages **checkpoints**, **retries**, and **error recovery**
- Runs **OPTIMIZE** and **VACUUM** in the background
- Tracks **data quality metrics** via expectations
- Generates **lineage** for governance and auditing

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">DLT Pipeline: Automatic DAG Generation</h3>
  <div style="display: flex; justify-content: center; align-items: center; gap: 8px; flex-wrap: wrap;">
    <div style="text-align: center;">
      <div style="background: #FFFFFF; border: 2px solid #E8E5E0; border-radius: 10px; padding: 16px 20px; min-width: 110px;">
        <div style="font-weight: bold; font-size: 13px; color: #1B1F24;">Source</div>
        <div style="font-size: 11px; color: #757575;">JSON in Volume</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #CD7F32;">&rarr;</div>
    <div style="text-align: center;">
      <div style="background: #FFF3E0; border: 3px solid #CD7F32; border-radius: 10px; padding: 16px 20px; min-width: 130px;">
        <div style="font-weight: bold; font-size: 14px; color: #CD7F32;">BRONZE</div>
        <div style="font-size: 11px; color: #1B1F24;">Streaming Table</div>
        <div style="font-size: 10px; color: #757575;">cloudFiles ingestion</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #9E9E9E;">&rarr;</div>
    <div style="text-align: center;">
      <div style="background: #F5F5F5; border: 3px solid #9E9E9E; border-radius: 10px; padding: 16px 20px; min-width: 130px;">
        <div style="font-weight: bold; font-size: 14px; color: #757575;">SILVER</div>
        <div style="font-size: 11px; color: #1B1F24;">Streaming Table</div>
        <div style="font-size: 10px; color: #757575;">Cleaned + Expectations</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #F9A825;">&rarr;</div>
    <div style="text-align: center;">
      <div style="background: #FFFDE7; border: 3px solid #F9A825; border-radius: 10px; padding: 16px 20px; min-width: 130px;">
        <div style="font-weight: bold; font-size: 14px; color: #F9A825;">GOLD</div>
        <div style="font-size: 11px; color: #1B1F24;">Materialized View</div>
        <div style="font-size: 10px; color: #757575;">Business Aggregates</div>
      </div>
    </div>
  </div>
  <p style="text-align: center; font-size: 12px; color: #757575; margin-top: 12px; margin-bottom: 0;">DLT builds this DAG automatically from your table references &mdash; no explicit dependency wiring needed.</p>
</div>
""")

DLT Pipeline: Automatic DAG Generation 
 
 
 
 Source 
 JSON in Volume 
 
 
 → 
 
 
 BRONZE 
 Streaming Table 
 cloudFiles ingestion 
 
 
 → 
 
 
 SILVER 
 Streaming Table 
 Cleaned + Expectations 
 
 
 → 
 
 
 GOLD 
 Materialized View 
 Business Aggregates 
 
 
 
 DLT builds this DAG automatically from your table references — no explicit dependency wiring needed.

In [1]:
displayHTML("""
<div style="background: #F1F8E9; border-left: 4px solid #00A972; padding: 14px 20px; border-radius: 6px; margin-top: 10px; font-family: 'Segoe UI', Arial, sans-serif; color: #1B1F24;">
  <strong style="color: #1B3A4B;">Key Takeaway:</strong> DLT is a <em>declarative</em> framework. You define <strong>what</strong> each table should contain (the query), and DLT handles the <strong>how</strong> (orchestration, checkpoints, retries, optimization). This reduces pipeline code by up to 70% compared to imperative ETL.
</div>
""")

Key Takeaway: DLT is a declarative framework. You define what each table should contain (the query), and DLT handles the how (orchestration, checkpoints, retries, optimization). This reduces pipeline code by up to 70% compared to imperative ETL.

---

## Section 2: DLT Python API Reference

Before writing the pipeline, here is a quick reference of the DLT Python API elements you will use:

| API Element | Purpose | Example |
|---|---|---|
| `import dlt` | Import the DLT module (available automatically in DLT runtime) | `import dlt` |
| `@dlt.table()` | Decorator that defines a DLT table or materialized view | `@dlt.table(name="my_table", comment="...")` |
| `dlt.read_stream("table")` | Read another DLT table as a **stream** (for streaming tables) | `dlt.read_stream("bronze")` |
| `dlt.read("table")` | Read another DLT table as a **batch** (for materialized views) | `dlt.read("silver")` |
| `@dlt.expect()` | **Warn** on quality violation (keep the row, log the metric) | `@dlt.expect("valid_id", "id IS NOT NULL")` |
| `@dlt.expect_or_drop()` | **Drop** rows that violate the quality rule | `@dlt.expect_or_drop("positive_val", "value > 0")` |
| `@dlt.expect_or_fail()` | **Fail** the entire pipeline if any row violates the rule | `@dlt.expect_or_fail("no_nulls", "key IS NOT NULL")` |

---

## Section 3: Configuration

Define paths and constants used throughout the pipeline. In DLT, you can use regular Python variables — they are evaluated when the pipeline starts.

In [ ]:
import dlt
from pyspark.sql.functions import (
    col, current_timestamp, input_file_name, expr,
    count, sum as spark_sum, avg, round as spark_round,
    year, month, to_date, when, lit
)

# ============================================================
# CONFIGURATION
# ============================================================
# Path to the raw placements JSON data in the training Volume
# This is the same data used in Labs 04-06

CATALOG_NAME = "axa_training"  # Change to your catalog name
SCHEMA_ASSETS = "lab_assets"
VOLUME_NAME = "training_volume"

RAW_PLACEMENTS_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_ASSETS}/{VOLUME_NAME}/raw_data/placements"
RAW_ENTITIES_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_ASSETS}/{VOLUME_NAME}/raw_data/entities"

---

## Section 4: Bronze Layer — Streaming Ingestion

The Bronze table ingests raw JSON files using **Auto Loader** (`cloudFiles`). In DLT, Auto Loader is even simpler than in Labs 04–06 because DLT manages checkpoints and schema tracking automatically.

In [1]:
displayHTML("""
<div style="display: flex; gap: 20px; flex-wrap: wrap; margin: 20px 0; font-family: 'Cascadia Code', Consolas, monospace; font-size: 13px;">
  <div style="flex: 1; min-width: 280px; background: #1B3A4B; color: #F9F7F4; border-radius: 10px; padding: 18px; line-height: 1.7;">
    <div style="color: #FF6F47; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">Manual Auto Loader (Lab 04)</div>
    spark.readStream<br>
    &nbsp;&nbsp;.format(<span style="color: #81C784;">"cloudFiles"</span>)<br>
    &nbsp;&nbsp;.option(<span style="color: #81C784;">"cloudFiles.format"</span>, ...)<br>
    &nbsp;&nbsp;.option(<span style="color: #81C784;">"schemaLocation"</span>, ...)<br>
    &nbsp;&nbsp;.load(path)<br><br>
    <span style="color: #FF6F47;"># + writeStream with checkpoint</span><br>
    <span style="color: #FF6F47;"># + checkpoint management</span><br>
    <span style="color: #FF6F47;"># + error handling</span>
  </div>
  <div style="flex: 1; min-width: 280px; background: #00A972; color: #FFFFFF; border-radius: 10px; padding: 18px; line-height: 1.7;">
    <div style="color: #FFFDE7; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">DLT Auto Loader</div>
    spark.readStream<br>
    &nbsp;&nbsp;.format(<span style="color: #C8E6C9;">"cloudFiles"</span>)<br>
    &nbsp;&nbsp;.option(<span style="color: #C8E6C9;">"cloudFiles.format"</span>, ...)<br>
    &nbsp;&nbsp;.load(path)<br><br>
    <span style="color: #FFFDE7;"># No schemaLocation needed!</span><br>
    <span style="color: #FFFDE7;"># No writeStream needed!</span><br>
    <span style="color: #FFFDE7;"># No checkpoint management!</span><br>
    <span style="color: #C8E6C9;"># DLT handles everything.</span>
  </div>
</div>
""")

Manual Auto Loader (Lab 04) 
 spark.readStream 
   .format( "cloudFiles" ) 
   .option( "cloudFiles.format" , ...) 
   .option( "schemaLocation" , ...) 
   .load(path) 
 # + writeStream with checkpoint 
 # + checkpoint management 
 # + error handling 
 
 
 DLT Auto Loader 
 spark.readStream 
   .format( "cloudFiles" ) 
   .option( "cloudFiles.format" , ...) 
   .load(path) 
 # No schemaLocation needed! 
 # No writeStream needed! 
 # No checkpoint management! 
 # DLT handles everything.

We also add audit columns (`_ingestion_time`, `_source_file`) just like in Lab 04, so we maintain data lineage.

In [ ]:
# ============================================================
# BRONZE: Raw placements ingestion via Auto Loader
# ============================================================

@dlt.table(
    name="bronze_placements",
    comment="Bronze: Raw placements data ingested from training volume via Auto Loader"
)
def bronze_placements():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(RAW_PLACEMENTS_PATH)
        .withColumn("_ingestion_time", current_timestamp())
        .withColumn("_source_file", input_file_name())
    )

In [1]:
displayHTML("""
<div style="background: #F1F8E9; border-left: 4px solid #00A972; padding: 14px 20px; border-radius: 6px; margin-top: 10px; font-family: 'Segoe UI', Arial, sans-serif; color: #1B1F24;">
  <strong style="color: #1B3A4B;">Key Takeaway:</strong> In DLT, Auto Loader needs only the <code>format</code> and <code>load(path)</code>. DLT automatically manages the schema location, checkpoints, and write destination. Compare this to the 15+ lines you wrote in Lab 04 for the same result.
</div>
""")

Key Takeaway: In DLT, Auto Loader needs only the format and load(path) . DLT automatically manages the schema location, checkpoints, and write destination. Compare this to the 15+ lines you wrote in Lab 04 for the same result.

---

## Section 5: Silver Layer — Data Quality with Expectations

The Silver table reads from Bronze, applies data quality rules, and cleans the data. In Labs 04–06, you wrote `.filter()` chains to remove bad rows. With DLT, you declare **expectations** instead:

| Expectation | SQL Expression | Policy | Effect |
|---|---|---|---|
| `@dlt.expect()` | `"market_value IS NOT NULL"` | **Warn** | Keep the row, log the violation |
| `@dlt.expect_or_drop()` | `"market_value > 0"` | **Drop** | Remove the row silently |
| `@dlt.expect_or_fail()` | `"placement_id IS NOT NULL"` | **Fail** | Stop the entire pipeline |

This gives you **built-in data quality monitoring** — the DLT dashboard shows exactly how many rows passed or failed each expectation.

### Exercise: Complete the Silver Table

**Fill in the blanks** to:
1. Add an expectation that **drops** rows where `market_value` is not positive
2. Add an expectation that **warns** (but keeps) rows where `book_value` is null
3. Read from the bronze table using the DLT streaming reader
4. Cast `valuation_date` to a proper date type

In [ ]:
# ============================================================
# SILVER: Cleaned and validated placements
# ============================================================

@dlt.table(
    name="silver_placements",
    comment="Silver: Cleaned placements with data quality expectations"
)
@dlt.expect_or_fail("valid_placement_id", "placement_id IS NOT NULL")
@dlt.________("positive_market_value", "market_value > 0")          # TODO: drop rows with non-positive market_value
@dlt.________("has_book_value", "book_value IS NOT NULL")            # TODO: warn (but keep) rows with null book_value
def silver_placements():
    return (
        dlt.________("bronze_placements")                             # TODO: read bronze as a stream
        .withColumn("valuation_date", ________(col("valuation_date")))  # TODO: cast string to date
        .withColumn("valuation_year", year(col("valuation_date")))
        .withColumn("valuation_month", month(col("valuation_date")))
        .withColumn(
            "unrealized_gain_loss",
            col("market_value") - col("book_value")
        )
        .withColumn(
            "gain_loss_pct",
            spark_round(
                (col("market_value") - col("book_value")) / col("book_value") * 100,
                2
            )
        )
        .select(
            "placement_id",
            "axa_entity_id",
            "asset_class",
            "instrument_id",
            "market_value",
            "book_value",
            "unrealized_gain_loss",
            "gain_loss_pct",
            "currency",
            "valuation_date",
            "valuation_year",
            "valuation_month",
            "_ingestion_time",
            "_source_file"
        )
    )

In [1]:
displayHTML("""
<details>
  <summary><strong>Click to reveal more hints</strong></summary>

**Hints:**
- The first blank: you want rows with non-positive `market_value` to be **removed** — which expectation policy drops bad rows? Look at the API reference table in Section 2
- The second blank: you want to **log** a warning for null `book_value` but still keep the row — which policy just warns without taking action?
- The third blank: Bronze is a streaming table, so you need the **streaming** reader from the DLT module — its name is `read_` + the type of processing
- The fourth blank: you used this function in Lab 05 to cast a string column to a `DATE` type — two words joined: "to" + "date"

</details>
""")

Click to reveal more hints 

**Hints:**
- The first blank: you want rows with non-positive `market_value` to be **removed** — which expectation policy drops bad rows? Look at the API reference table in Section 2
- The second blank: you want to **log** a warning for null `book_value` but still keep the row — which policy just warns without taking action?
- The third blank: Bronze is a streaming table, so you need the **streaming** reader from the DLT module — its name is `read_` + the type of processing
- The fourth blank: you used this function in Lab 05 to cast a string column to a `DATE` type — two words joined: "to" + "date"

In [1]:
displayHTML("""
<div style="background: #F1F8E9; border-left: 4px solid #00A972; padding: 14px 20px; border-radius: 6px; margin-top: 10px; font-family: 'Segoe UI', Arial, sans-serif; color: #1B1F24;">
  <strong style="color: #1B3A4B;">Key Takeaway:</strong> DLT expectations replace the manual <code>.filter()</code> chains from Lab 05. The three policies (<code>expect</code> = warn, <code>expect_or_drop</code> = drop, <code>expect_or_fail</code> = fail) give you fine-grained control. All metrics are tracked automatically in the DLT dashboard.
</div>
""")

Key Takeaway: DLT expectations replace the manual .filter() chains from Lab 05. The three policies ( expect = warn, expect_or_drop = drop, expect_or_fail = fail) give you fine-grained control. All metrics are tracked automatically in the DLT dashboard.

---

## Section 6: Gold Layer — Business Aggregations

The Gold table creates pre-aggregated business metrics. In DLT, Gold tables are typically **materialized views** — they read from Silver using `dlt.read()` (batch) instead of `dlt.read_stream()` (streaming).

In [1]:
displayHTML("""
<div style="display: flex; gap: 20px; flex-wrap: wrap; margin: 20px 0; font-family: 'Segoe UI', sans-serif; font-size: 14px;">
  <div style="flex: 1; min-width: 260px; background: #1B3A4B; color: #FFFFFF; border-radius: 10px; padding: 18px;">
    <div style="font-weight: 700; color: #FF6F47; margin-bottom: 8px;">Streaming Table (Bronze, Silver)</div>
    <ul style="margin: 0; padding-left: 18px; line-height: 1.9; color: #F9F7F4;">
      <li>Processes data <strong>incrementally</strong></li>
      <li>Uses <code style="background: rgba(255,255,255,0.15); padding: 2px 6px; border-radius: 4px;">dlt.read_stream()</code></li>
      <li>Append-only</li>
      <li>Ideal for ingestion &amp; cleaning</li>
    </ul>
  </div>
  <div style="flex: 1; min-width: 260px; background: #00A972; color: #FFFFFF; border-radius: 10px; padding: 18px;">
    <div style="font-weight: 700; color: #FFFDE7; margin-bottom: 8px;">Materialized View (Gold)</div>
    <ul style="margin: 0; padding-left: 18px; line-height: 1.9; color: #FFFFFF;">
      <li>Recomputed on <strong>each pipeline run</strong></li>
      <li>Uses <code style="background: rgba(255,255,255,0.15); padding: 2px 6px; border-radius: 4px;">dlt.read()</code></li>
      <li>Full refresh of aggregates</li>
      <li>Ideal for aggregations &amp; reporting</li>
    </ul>
  </div>
</div>
""")

Streaming Table (Bronze, Silver) 
 
 Processes data incrementally 
 Uses dlt.read_stream() 
 Append-only 
 Ideal for ingestion & cleaning 
 
 
 
 Materialized View (Gold) 
 
 Recomputed on each pipeline run 
 Uses dlt.read() 
 Full refresh of aggregates 
 Ideal for aggregations & reporting

### Exercise: Complete the Gold Table

**Fill in the blanks** to:
1. Read from the Silver table using the DLT **batch** reader
2. Group by `asset_class`
3. Calculate the total market value

In [ ]:
# ============================================================
# GOLD: Asset allocation summary (materialized view)
# ============================================================

@dlt.table(
    name="gold_asset_summary",
    comment="Gold: Asset allocation summary by asset class"
)
def gold_asset_summary():
    return (
        dlt.________("silver_placements")                              # TODO: read silver as batch (not stream)
        .groupBy("________")                                           # TODO: group by asset class
        .agg(
            ________("market_value").alias("total_market_value"),       # TODO: sum of market_value
            count("*").alias("position_count"),
            spark_round(avg("market_value"), 2).alias("avg_position_value"),
            spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct")
        )
    )

In [1]:
displayHTML("""
<details>
  <summary><strong>Click to reveal more hints</strong></summary>

**Hints:**
- The first blank: Gold tables are materialized views that read in **batch** mode — the DLT function is simply `read()` (no "stream" suffix)
- The second blank: you are building an asset allocation summary — the column that categorizes investments by type is `asset_class`
- The third blank: you need the sum function — remember the aliased import `spark_sum` from the imports cell

</details>
""")

Click to reveal more hints 

**Hints:**
- The first blank: Gold tables are materialized views that read in **batch** mode — the DLT function is simply `read()` (no "stream" suffix)
- The second blank: you are building an asset allocation summary — the column that categorizes investments by type is `asset_class`
- The third blank: you need the sum function — remember the aliased import `spark_sum` from the imports cell

In [1]:
displayHTML("""
<div style="background: #F1F8E9; border-left: 4px solid #00A972; padding: 14px 20px; border-radius: 6px; margin-top: 10px; font-family: 'Segoe UI', Arial, sans-serif; color: #1B1F24;">
  <strong style="color: #1B3A4B;">Key Takeaway:</strong> Use <code>dlt.read_stream()</code> for streaming tables (Bronze, Silver) and <code>dlt.read()</code> for materialized views (Gold aggregations). DLT determines the table type from how you read and return data.
</div>
""")

Key Takeaway: Use dlt.read_stream() for streaming tables (Bronze, Silver) and dlt.read() for materialized views (Gold aggregations). DLT determines the table type from how you read and return data.

---

## Section 7: Entity Performance Table (Bonus)

Let's add a second Gold table that summarizes performance by AXA entity. This demonstrates that a single DLT pipeline can produce **multiple Gold tables** from the same Silver source — DLT handles the DAG automatically.

In [ ]:
# ============================================================
# GOLD: Entity performance summary (materialized view)
# ============================================================

@dlt.table(
    name="gold_entity_performance",
    comment="Gold: Performance summary by AXA entity"
)
def gold_entity_performance():
    return (
        dlt.read("silver_placements")
        .groupBy("axa_entity_id")
        .agg(
            spark_sum("market_value").alias("total_aum"),
            count("*").alias("position_count"),
            spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct"),
            spark_sum("unrealized_gain_loss").alias("total_gain_loss")
        )
    )

In [1]:
displayHTML("""
<div style="background: #F1F8E9; border-left: 4px solid #00A972; padding: 14px 20px; border-radius: 6px; margin-top: 10px; font-family: 'Segoe UI', Arial, sans-serif; color: #1B1F24;">
  <strong style="color: #1B3A4B;">Key Takeaway:</strong> Multiple Gold tables can read from the same Silver table. DLT detects the shared dependency and runs the Silver transformation once, then fans out to both Gold tables. This is the power of the declarative DAG.
</div>
""")

Key Takeaway: Multiple Gold tables can read from the same Silver table. DLT detects the shared dependency and runs the Silver transformation once, then fans out to both Gold tables. This is the power of the declarative DAG.

---

## Section 8: Create and Run the Pipeline

Now that the notebook is complete, you need to create a **DLT pipeline** in the Databricks UI that uses this notebook as its source.

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border: 2px solid #FF3621; padding: 20px; border-radius: 10px; margin: 15px 0; font-family: 'Segoe UI', Arial, sans-serif;">

<div style="font-weight: bold; font-size: 15px; color: #FF3621; margin-bottom: 12px;">Step-by-Step: Create the DLT Pipeline</div>

<ol style="margin: 0; padding-left: 20px; line-height: 2.2; color: #1B1F24;">
  <li>In the left sidebar, click <strong>Jobs &amp; Pipelines</strong>, then select the <strong>Delta Live Tables</strong> tab</li>
  <li>Click <strong>Create Pipeline</strong></li>
  <li>Configure the pipeline:
    <table style="margin: 10px 0; border-collapse: collapse; font-size: 13px; width: 100%;">
      <tr style="background: #F9F7F4;"><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><strong>Pipeline name</strong></td><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><code>DLT_Medallion_Lab</code></td></tr>
      <tr><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><strong>Product edition</strong></td><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><code>Advanced</code></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><strong>Pipeline mode</strong></td><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><code>Triggered</code> (runs once when started)</td></tr>
      <tr><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><strong>Source code</strong></td><td style="padding: 8px 12px; border: 1px solid #E8E5E0;">Browse to this notebook: <code>/labs/jour2/07_DLT_Pipeline</code></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><strong>Target schema</strong></td><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><code>dlt_lab</code> (DLT will create this schema)</td></tr>
      <tr><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><strong>Target catalog</strong></td><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><code>axa_training</code> (Unity Catalog)</td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 8px 12px; border: 1px solid #E8E5E0;"><strong>Cluster mode</strong></td><td style="padding: 8px 12px; border: 1px solid #E8E5E0;">Use default (DLT manages its own compute)</td></tr>
    </table>
  </li>
  <li>Click <strong>Create</strong></li>
  <li>Click <strong>Start</strong> to run the pipeline</li>
  <li>Watch the DAG build in real time — you should see: <code>bronze_placements</code> &rarr; <code>silver_placements</code> &rarr; <code>gold_asset_summary</code> + <code>gold_entity_performance</code></li>
</ol>

</div>
""")

Pipeline name,DLT_Medallion_Lab
Product edition,Advanced
Pipeline mode,Triggered (runs once when started)
Source code,Browse to this notebook: /labs/jour2/07_DLT_Pipeline
Target schema,dlt_lab (DLT will create this schema)
Target catalog,axa_training (Unity Catalog)
Cluster mode,Use default (DLT manages its own compute)


### What If the Pipeline Fails?

If the pipeline fails, it is likely because one of the blanks was not filled in correctly.

1. Click on the **failed table** in the DAG to see the error message
2. Come back to this notebook and fix the code
3. Go back to the pipeline page and click **Start** again
4. DLT will automatically pick up your changes

> **Tip:** The most common errors are:
> - `NameError` — a blank was not filled in (still contains `________`)
> - `AttributeError: module 'dlt' has no attribute '...'` — check the function name spelling
> - `AnalysisException` — check column names in your expectations or select

---

## Section 9: Monitor Pipeline Quality

Once the pipeline completes successfully, explore the DLT dashboard:

### 9.1 — DAG Visualization

The pipeline page shows a **DAG** (Directed Acyclic Graph) of all your tables. Click on any table to see:
- Number of records processed
- Processing time
- Schema details

### 9.2 — Data Quality Metrics

Click on the **Silver table** in the DAG. You should see the expectations panel showing:

| Expectation | Policy | Records Passed | Records Failed |
|---|---|---|---|
| `valid_placement_id` | FAIL | All | 0 |
| `positive_market_value` | DROP | Most | Some dropped |
| `has_book_value` | WARN | Most | Some warned |

This is the **data quality observability** that DLT provides out of the box — no extra code needed.

### 9.3 — Query the Results

After the pipeline completes, the tables are available in Unity Catalog. You can query them from any notebook:

```sql
-- Query the Gold asset summary (run this in a separate notebook or SQL editor)
SELECT * FROM axa_training.dlt_lab.gold_asset_summary ORDER BY total_market_value DESC;

-- Check data quality metrics from the event log
SELECT * FROM axa_training.dlt_lab.gold_entity_performance ORDER BY total_aum DESC;

-- Count records at each layer
SELECT 'Bronze' AS layer, COUNT(*) AS records FROM axa_training.dlt_lab.bronze_placements
UNION ALL
SELECT 'Silver', COUNT(*) FROM axa_training.dlt_lab.silver_placements
UNION ALL
SELECT 'Gold (Assets)', COUNT(*) FROM axa_training.dlt_lab.gold_asset_summary
UNION ALL
SELECT 'Gold (Entities)', COUNT(*) FROM axa_training.dlt_lab.gold_entity_performance;
```

In [1]:
displayHTML("""
<div style="background: #F1F8E9; border-left: 4px solid #00A972; padding: 14px 20px; border-radius: 6px; margin-top: 10px; font-family: 'Segoe UI', Arial, sans-serif; color: #1B1F24;">
  <strong style="color: #1B3A4B;">Key Takeaway:</strong> DLT provides built-in observability: the DAG shows data flow, expectations show quality metrics, and the event log records every operation. This replaces custom monitoring code you would otherwise have to build yourself.
</div>
""")

Key Takeaway: DLT provides built-in observability: the DAG shows data flow, expectations show quality metrics, and the event log records every operation. This replaces custom monitoring code you would otherwise have to build yourself.

---

## Section 10: DLT vs Manual ETL Comparison

Now that you have built the same Medallion pipeline both ways, here is a side-by-side comparison:

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">

<table style="width: 100%; border-collapse: collapse; font-size: 13px;">
<tr style="background: #1B3A4B; color: #FFFFFF;">
  <th style="padding: 12px; text-align: left; width: 30%;">Aspect</th>
  <th style="padding: 12px; text-align: left; width: 35%;">Manual ETL (Labs 04–06)</th>
  <th style="padding: 12px; text-align: left; width: 35%;">DLT (Lab 07)</th>
</tr>
<tr style="background: #FFFFFF;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Code volume</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">3 notebooks, ~300 lines each</td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">1 notebook, ~60 lines of pipeline code</td>
</tr>
<tr style="background: #F9F7F4;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Checkpoints</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Manual: configure path, manage state</td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Automatic: DLT handles everything</td>
</tr>
<tr style="background: #FFFFFF;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Data quality</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Manual: .filter() chains, custom logging</td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Declarative: @dlt.expect decorators with built-in dashboard</td>
</tr>
<tr style="background: #F9F7F4;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>OPTIMIZE / VACUUM</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Manual: run SQL commands (Lab 06)</td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Automatic: runs in the background</td>
</tr>
<tr style="background: #FFFFFF;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Error recovery</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Manual: re-run failed notebooks</td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Automatic: retry logic + partial re-execution</td>
</tr>
<tr style="background: #F9F7F4;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Orchestration</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Manual: Workflows with task dependencies</td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Automatic: DAG inferred from table references</td>
</tr>
<tr style="background: #FFFFFF;">
  <td style="padding: 10px; color: #1B1F24;"><strong>Lineage</strong></td>
  <td style="padding: 10px; color: #1B1F24;">Manual: document it yourself</td>
  <td style="padding: 10px; color: #1B1F24;">Automatic: visual DAG with data flow metrics</td>
</tr>
</table>

</div>
""")

Aspect,Manual ETL (Labs 04–06),DLT (Lab 07)
Code volume,"3 notebooks, ~300 lines each","1 notebook, ~60 lines of pipeline code"
Checkpoints,"Manual: configure path, manage state",Automatic: DLT handles everything
Data quality,"Manual: .filter() chains, custom logging",Declarative: @dlt.expect decorators with built-in dashboard
OPTIMIZE / VACUUM,Manual: run SQL commands (Lab 06),Automatic: runs in the background
Error recovery,Manual: re-run failed notebooks,Automatic: retry logic + partial re-execution
Orchestration,Manual: Workflows with task dependencies,Automatic: DAG inferred from table references
Lineage,Manual: document it yourself,Automatic: visual DAG with data flow metrics


---

## What You Learned

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Lab 07 Summary</h3>
  <table style="width: 100%; border-collapse: collapse; background: #FFFFFF; border-radius: 8px; overflow: hidden;">
    <thead>
      <tr style="background: #1B3A4B; color: #FFFFFF;">
        <th style="padding: 12px 15px; text-align: left;">Concept</th>
        <th style="padding: 12px 15px; text-align: left;">What You Learned</th>
        <th style="padding: 12px 15px; text-align: left;">Key API</th>
      </tr>
    </thead>
    <tbody>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">DLT Basics</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Declarative pipeline definitions replace imperative ETL code</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>import dlt</code>, <code>@dlt.table()</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Bronze (Streaming)</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Auto Loader ingestion without manual checkpoints</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>spark.readStream.format("cloudFiles")</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #757575;">Silver (Quality)</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Data quality expectations with warn/drop/fail policies</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>@dlt.expect_or_drop()</code>, <code>dlt.read_stream()</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #F9A825;">Gold (Views)</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Materialized views for pre-aggregated business metrics</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>dlt.read()</code>, <code>.groupBy().agg()</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #1B3A4B;">Observability</td>
        <td style="padding: 10px 15px; color: #1B1F24;">DAG visualization, quality metrics, and event logging</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Built-in DLT dashboard</td>
      </tr>
      <tr>
        <td style="padding: 10px 15px; font-weight: bold; color: #1B3A4B;">Operations</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Automatic OPTIMIZE, VACUUM, error recovery, and scaling</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Managed by DLT runtime</td>
      </tr>
    </tbody>
  </table>
</div>
""")

Concept,What You Learned,Key API
DLT Basics,Declarative pipeline definitions replace imperative ETL code,"import dlt, @dlt.table()"
Bronze (Streaming),Auto Loader ingestion without manual checkpoints,"spark.readStream.format(""cloudFiles"")"
Silver (Quality),Data quality expectations with warn/drop/fail policies,"@dlt.expect_or_drop(), dlt.read_stream()"
Gold (Views),Materialized views for pre-aggregated business metrics,"dlt.read(), .groupBy().agg()"
Observability,"DAG visualization, quality metrics, and event logging",Built-in DLT dashboard
Operations,"Automatic OPTIMIZE, VACUUM, error recovery, and scaling",Managed by DLT runtime


In [1]:
displayHTML("""
<div style="background: #1B3A4B; padding: 30px; border-radius: 12px; margin: 20px 0; font-family: monospace; color: #FFFFFF; border: 2px solid #00A972;">

<div style="text-align: center; font-size: 18px; font-weight: bold; color: #00A972; margin-bottom: 20px;">COURSE COMPLETE</div>

<div style="text-align: center; color: #F9F7F4; font-size: 14px; line-height: 1.8;">
  Over the past two days, you have built the same Medallion pipeline <strong>two ways</strong>:<br>
  <strong>Imperative</strong> (Labs 04–06) &mdash; full control, full responsibility<br>
  <strong>Declarative</strong> (Lab 07) &mdash; same result, fraction of the code<br><br>
  Understanding both approaches makes you a well-rounded Databricks engineer:<br>
  use <strong>manual ETL</strong> when you need fine-grained control,<br>
  use <strong>DLT</strong> when you want productivity, quality, and operations handled for you.
</div>

</div>
""")

COURSE COMPLETE 

 
 Over the past two days, you have built the same Medallion pipeline two ways : 
 Imperative (Labs 04–06) — full control, full responsibility 
 Declarative (Lab 07) — same result, fraction of the code 
 Understanding both approaches makes you a well-rounded Databricks engineer: 
 use manual ETL when you need fine-grained control, 
 use DLT when you want productivity, quality, and operations handled for you.

---

## Cleanup

To clean up the DLT pipeline and its tables:

1. Go to **Jobs & Pipelines** in the left sidebar, then select the **Delta Live Tables** tab
2. Find your pipeline: `DLT_Medallion_Lab`
3. Click **Delete** to remove the pipeline and its compute
4. The tables in `axa_training.dlt_lab` will remain until you drop them:

```sql
-- Run in a separate notebook or SQL editor
DROP SCHEMA IF EXISTS axa_training.dlt_lab CASCADE;
```